# Task 5 — Restaurant Recommendation System

**Objective:** Build a content-based restaurant recommendation system that suggests  
relevant restaurants based on user preferences (cuisine, city, price, rating).

---
### Technique: TF-IDF + Cosine Similarity (Content-Based Filtering)

**Why content-based filtering?**  
We have rich restaurant attribute data but no user–restaurant interaction history  
(no purchase logs, no user profiles). Content-based filtering is the correct  
approach — it recommends restaurants similar to what the user describes.

**How it works:**
1. Each restaurant is converted into a text document combining cuisine, city, price tier, rating category, and service features
2. TF-IDF vectorises these documents (weighs rare but important terms higher)
3. The user's preferences become a query document
4. Cosine similarity ranks restaurants by how closely they match the query
5. A composite score (0.5 × similarity + 0.5 × normalised rating) reranks for quality

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_cleaning   import load_clean
from src.recommendation  import RestaurantRecommender
from src.analysis        import PRICE_LABELS

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print('Libraries loaded ✓')

## 1. Load Data & Build Recommender

In [ ]:
df = load_clean()
print(f"Dataset: {df.shape[0]:,} restaurants | {df['City'].nunique()} cities")

rec = RestaurantRecommender()
rec.fit(df)
print("\nRecommender system ready ✓")

## 2. Explore Available Options

In [ ]:
cities = rec.available_cities()
print(f"Available cities ({len(cities)}):")
print(cities[:30], '...' if len(cities) > 30 else '')

In [ ]:
cuisines = rec.available_cuisines()
print(f"Available primary cuisines ({len(cuisines)}):")
print(cuisines[:40], '...' if len(cuisines) > 40 else '')

## 3. Example 1 — Italian Mid-Range Restaurants

In [ ]:
print("=" * 60)
print("USER PREFERENCES:")
print("  Cuisine    : Italian")
print("  Price Range: Mid-Range (2)")
print("  Min Rating : 3.5")
print("  City       : Any")
print("=" * 60)

results1 = rec.recommend(
    cuisine='Italian',
    city='',
    price_range=2,
    min_rating=3.5,
    top_n=8
)
results1[['Restaurant Name', 'City', 'Cuisines',
          'Aggregate rating', 'Price Label',
          'Online Delivery', 'Composite Score']]

## 4. Example 2 — North Indian in New Delhi (Budget)

In [ ]:
print("=" * 60)
print("USER PREFERENCES:")
print("  Cuisine    : North Indian")
print("  City       : New Delhi")
print("  Price Range: Budget (1)")
print("  Min Rating : 3.5")
print("=" * 60)

results2 = rec.recommend(
    cuisine='North Indian',
    city='New Delhi',
    price_range=1,
    min_rating=3.5,
    top_n=8
)
results2[['Restaurant Name', 'City', 'Cuisines',
          'Aggregate rating', 'Votes', 'Price Label',
          'Online Delivery', 'Table Booking', 'Composite Score']]

## 5. Example 3 — Best Restaurants in New Delhi (Any Cuisine)

In [ ]:
print("=" * 60)
print("USER PREFERENCES:")
print("  City       : New Delhi")
print("  Min Rating : 4.0")
print("  Require Online Delivery: Yes")
print("=" * 60)

results3 = rec.recommend(
    cuisine='',
    city='New Delhi',
    price_range=0,
    min_rating=4.0,
    top_n=8,
    require_delivery=True
)
results3[['Restaurant Name', 'City', 'Primary Cuisine',
          'Aggregate rating', 'Votes', 'Price Label',
          'Average Cost for two', 'Online Delivery']]

## 6. Example 4 — Japanese Luxury with Table Booking

In [ ]:
print("=" * 60)
print("USER PREFERENCES:")
print("  Cuisine    : Japanese")
print("  Price Range: Luxury (4)")
print("  Min Rating : 4.0")
print("  Table Booking: Required")
print("=" * 60)

results4 = rec.recommend(
    cuisine='Japanese',
    city='',
    price_range=4,
    min_rating=4.0,
    top_n=8,
    require_booking=True
)
if results4.empty:
    # Relax constraints if no results
    print('No results with all constraints — relaxing price range...')
    results4 = rec.recommend(
        cuisine='Japanese', city='', price_range=3,
        min_rating=4.0, top_n=8, require_booking=True
    )

results4[['Restaurant Name', 'City', 'Cuisines',
          'Aggregate rating', 'Votes', 'Price Label',
          'Average Cost for two', 'Table Booking']]

## 7. Top-Rated Restaurants in a City (Quick Lookup)

In [ ]:
print("Top 5 Restaurants in New Delhi by Rating:")
rec.top_rated_in_city('New Delhi', top_n=5)

## 8. Visualise Recommendation Scores

In [ ]:
# Visualise the top-8 Italian recommendations
if not results1.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = sns.color_palette('viridis', len(results1))
    ax.barh(results1['Restaurant Name'], results1['Composite Score'],
            color=colors)
    ax.set_xlabel('Composite Score (similarity + rating)', fontsize=12)
    ax.set_title('Top Italian Mid-Range Recommendations — Composite Score',
                 fontsize=13, fontweight='bold')
    ax.set_xlim(0, 1)
    for i, val in enumerate(results1['Composite Score']):
        ax.text(val + 0.01, i, f'{val:.3f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# Scatter: similarity score vs rating for recommendations
if not results2.empty and 'Similarity Score' in results2.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(results2['Similarity Score'], results2['Aggregate rating'],
               s=120, color='steelblue', edgecolors='black', linewidth=0.8, zorder=5)
    for _, row in results2.iterrows():
        ax.annotate(row['Restaurant Name'],
                    (row['Similarity Score'], row['Aggregate rating']),
                    fontsize=7, ha='left', va='bottom',
                    xytext=(3, 3), textcoords='offset points')
    ax.set_xlabel('Cosine Similarity Score', fontsize=12)
    ax.set_ylabel('Aggregate Rating', fontsize=12)
    ax.set_title('Recommendations — Similarity vs Rating\n(North Indian, New Delhi, Budget)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Interactive Preference Demo

In [ ]:
# ── CHANGE THESE PREFERENCES AND RE-RUN THE CELL ──────────────────────────────
MY_CUISINE     = 'Chinese'     # e.g. 'North Indian', 'Italian', 'Chinese', 'Fast Food'
MY_CITY        = ''            # e.g. 'New Delhi', 'Gurgaon', '' for any city
MY_PRICE_RANGE = 2             # 1=Budget, 2=Mid-Range, 3=Premium, 4=Luxury, 0=any
MY_MIN_RATING  = 3.5           # 0 to 5
MY_TOP_N       = 10            # how many recommendations
NEED_DELIVERY  = False         # True if you need online delivery
NEED_BOOKING   = False         # True if you need table booking
# ─────────────────────────────────────────────────────────────────────────────

my_results = rec.recommend(
    cuisine=MY_CUISINE,
    city=MY_CITY,
    price_range=MY_PRICE_RANGE,
    min_rating=MY_MIN_RATING,
    top_n=MY_TOP_N,
    require_delivery=NEED_DELIVERY,
    require_booking=NEED_BOOKING,
)

if my_results.empty:
    print('No results found. Try relaxing your filters.')
else:
    print(f"\nTop {len(my_results)} recommendations for you:\n")
    display(my_results[['Restaurant Name','City','Cuisines','Aggregate rating',
                         'Price Label','Online Delivery','Table Booking',
                         'Average Cost for two','Composite Score']])

## 10. How to Interpret the Recommendations

```
UNDERSTANDING THE OUTPUT
=========================

Composite Score
  = 0.5 × Cosine Similarity  +  0.5 × Normalised Rating
  Range: 0.0 (worst) to 1.0 (perfect match)
  A score of 0.7+ indicates a very strong match.

Cosine Similarity
  Measures how closely the restaurant's content (cuisine, city, price,
  services) matches the user's query.
  1.0 = identical content description
  0.0 = no overlap at all

WHY THIS RESTAURANT IS RELEVANT
  Each recommendation is returned because it:
  1. Matches your specified cuisine
  2. Is in your requested city (or any city if none specified)
  3. Falls within your price range
  4. Meets your minimum rating threshold
  5. Offers delivery/booking if you required it

SYSTEM LIMITATIONS
  - Content-based systems do not personalise to user history
  - All users with the same query get the same results
  - A collaborative filtering layer (user ratings history) would improve
    personalisation — recommended as a future enhancement

FOR THE STREAMLIT APP
  Run: streamlit run app.py
  The app provides a GUI version of this recommendation system with
  dropdown menus, sliders, and a results table.
```